## CNN for Fossil Classification

In [1]:
# Import dependencies
import numpy as np 
import keras
from keras import layers
import tensorflow_datasets as tfds
from tensorflow import data as tf_data
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import random
from skimage import io
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


First, we import and split the data into testing, training, and validation sets. This should be written as a function so we can test on different folds during cross validation, where we will tune hyperparameters. 

In [3]:
image_samples = pd.read_csv("/content/drive/MyDrive/MLGeoProject/ImageSamples.csv")
header_name = 'samples'

# Takes in a list of samples (csv) and a target opb, opc, etc and returns a random sample as a string
def GetRandomThinSections(target, samplelist, headername):
    target = str(target)

    # Filter list of samples so we only deal with those involving target
    filtered_samples = samplelist[samplelist[headername].str.contains(target)]
    
    #randomly select one row (ie one thin section)
    if not filtered_samples.empty:
        random_thin_section = filtered_samples.sample(n=1).to_string(index=False, header=False) # Select 1 random row
    else:
        print(f"No rows found with '{target}' in 'samples'")
    return random_thin_section


In [4]:
#TO DO: Edit so we also get labels, wrap model in a function, write k fold CV for params
# 
#  For each opx (x = b thru f) we will hold back one total thin section and all of its related patches for testing data

# This function takes in a folder of patches and pre-determined sections to remove and splits into train, test, val data
def TrainTestValSplit(image_folder, sections):
    #image_folder = input_patches
    # randomly select a thin section (thin_section_nums)
    #thin_sections = random.sample(thin_section_nums, num_sections)
    #thin_sections = [str(ts) for ts in thin_sections]

    # Initialize empty lists
    test_data_names = []
    validation_data_names = []
    training_data_names = []
    remaining_data_names = []
    remaining_data = []
    test_data = []
    train_data = []
    val_data = []

    # Loop through targets to pull one random thin section and add to test data
    for section in sections:
        # Add thin sections corresponding to random choice to test data, then split remaining data into test and validation sets
        for filename in os.listdir(image_folder):
            if filename.lower().endswith((".jpg", ".png", ".jpeg")):
                filepath = os.path.join(image_folder, filename)
            if section in filename:
                test_data_names.append(filepath)
            else:
                remaining_data_names.append(filepath)
    # We now have the names of the files we want.
    # Now, split remaining data into training and validation sets at random
    print(test_data_names)
    random.shuffle(remaining_data_names) #randomly shuffles remaining image patches
    split = int(len(remaining_data_names) * 0.8)
    training_data_names = remaining_data_names[:split]
    validation_data_names = remaining_data_names[split:]

    # Read in images to add to data
    training_data = [io.imread(im) for im in training_data_names]
    validation_data = [io.imread(im) for im in validation_data_names]
    test_data = [io.imread(im) for im in test_data_names]

    #Stack into arrays so keras can read
    test_data = np.stack(test_data)
    validation_data = np.stack(validation_data)
    training_data = np.stack(training_data)
    return training_data, validation_data, test_data


center_folder = "/content/drive/MyDrive/MLGeoProject/Center_Fossil"
fossil_nocenter_folder = "/content/drive/MyDrive/MLGeoProject/No_Center_Fossil"
no_fossil_folder = "/content/drive/MyDrive/MLGeoProject/No_Fossil"
targets = ["opb", "opc", "opd", "ope", "opf"]
sections = []

#This randomly gets test sections for our data (they will be fixed for each run of traintestvalsplit)
for target in targets:
    test_section = GetRandomThinSections(target, image_samples, header_name)
    sections.append(test_section)
print("Thin Sections Pulled for Testing:")
print(sections)

train_center, val_center, test_center = TrainTestValSplit(center_folder, sections)
train_nocenter, val_nocenter, test_nocenter = TrainTestValSplit(fossil_nocenter_folder, sections)
train_nofossil, val_nofossil, test_nofossil = TrainTestValSplit(no_fossil_folder, sections)

# Now, get label data setting fossils to 1 and no fossil to 0

Y_train = np.concatenate([np.ones(train_center.shape[0]), np.zeros(train_nocenter.shape[0]), np.zeros(train_nofossil.shape[0])])
Y_val = np.concatenate([np.ones(val_center.shape[0]), np.zeros(val_nocenter.shape[0]), np.zeros(val_nofossil.shape[0])])
Y_test = np.concatenate([np.ones(test_center.shape[0]), np.zeros(test_nocenter.shape[0]), np.zeros(test_nofossil.shape[0])])
training_data = np.concatenate([train_center, train_nocenter, train_nofossil])
validation_data = np.concatenate([val_center, val_nocenter, val_nofossil])
testing_data = np.concatenate([test_center, test_nocenter, test_nofossil])

Thin Sections Pulled for Testing:
['opb_34.1', 'opc_64.1', 'opd_49.0', 'ope_66.0', 'opf_5.0']
['/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_ope_66.0d_patch3.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_ope_66.0a_patch4.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_ope_66.0c_patch5.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_ope_66.0b_patch5.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_ope_66.0c_patch4.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_ope_66.0b_patch4.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_opf_5.0b_patch2.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_opf_5.0b_patch1.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_opf_5.0b_patch3.png', '/content/drive/MyDrive/MLGeoProject/Center_Fossil/resized_opf_5.0b_patch4.png']
['/content/drive/MyDrive/MLGeoProject/No_Center_Fossil/resized_ope_66.0a_patch3.png', '/c

KeyboardInterrupt: 

In [ ]:
# Transform numpy arrays to tf datasets
training_data = tf.data.Dataset.from_tensor_slices((training_data, Y_train))
validation_data = tf.data.Dataset.from_tensor_slices((validation_data, Y_val))
testing_data = tf.data.Dataset.from_tensor_slices((testing_data, Y_test))

# Verify we did this correctly by plotting
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(training_data.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(int(label))
    plt.axis("off")



InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

We will use transfer learning using different pre-trained models on the imagenet dataset. We first train the Xception model. 

In [ ]:
# Our images are already resized, but we could include optional resizing in the final model
# We start with data augmentation
augmentation_layers = [
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.25)
]

def data_augmentation(x):
    for layer in augmentation_layers:
        x = layer(x)
    return x

# Create augmented data - Check with Lucy about what we want to do here
augmented_data = training_data.map(lambda x, y: (data_augmentation(x), y))

training_data = training_data.concatenate(augmented_data)

# Visualize training data with augmented data
plt.figure(figsize=(10, 10))
for i, (image, label) in enumerate(training_data.take(9)):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(int(label))
    plt.axis("off")
print(augmented_data)

In [12]:
# Batch and prefetch data ?

batch_size = 36 # this could change

training_data = training_data.batch(batch_size).prefetch(tf_data.AUTOTUNE).cache()
validation_data = validation_data.batch(batch_size).prefetch(tf_data.AUTOTUNE).cache()
testing_data = testing_data.batch(batch_size).prefetch(tf_data.AUTOTUNE).cache()

In [13]:
# We will use transfer learning using different pre-trained models on the imagenet dataset

# Create the base model with the pre-trained imagenet weights
base_model = keras.applications.Xception(
    weights = 'imagenet',
    input_shape = (200, 200, 3),
    include_top = False) # Why do we not include imagenet at the top? 

# Freeze the base model weights (they will not be trained)

base_model.trainable = False

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [14]:
# Create a new model that goes on top of the base imagenet model
inputs = keras.Input(shape = (200, 200, 3))
# We should scale our inputs to be between -1 and 1 (or should we normalize to have mean 0 and std 1?)
scale_layer = keras.layers.Rescaling(scale = 1/127.5, offset = -1)
x = scale_layer(inputs)

# We want to keep batchnorm layers of base model in inference (predict) mode so we do not train of them. 
# Make sure that base model is running on inference mode

x = base_model(x, training = False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.1)(x)
outputs = keras.layers.Dense(1)(x)
model = keras.Model(inputs, outputs)

model.summary(show_trainable = True)
# I think we should play with the layers we use here. We should try the setup they have, but maybe add more or different layers depending on how well the model does. 

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 200, 200, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ rescaling (Rescaling)       │ (None, 200, 200, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ xception (Functional)       │ (None, 7, 7, 2048)    │ 20,861,480 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ global_average_pooling2d    │ (None, 2048)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout (Dropout)           │ (None, 2048)          │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense (Dense)               │ (None, 1)             │      2,049 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 20,863,529 (79.59 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 20,861,480 (79.58 MB)

In [15]:
# Train the new top layer of the model
model.compile(
    optimizer = keras.optimizers.Adam(), # play with this also
    loss = keras.losses.BinaryCrossentropy(from_logits = True), #check this also
    metrics = [keras.metrics.BinaryAccuracy()],
)

epochs = 2 # also play with this
print("Fitting top layer of model")
model.fit(training_data, epochs = epochs, validation_data = validation_data)

Fitting top layer of model
Epoch 1/2
95/95 ━━━━━━━━━━━━━━━━━━━━ 855s 9s/step - binary_accuracy: 0.8941 - loss: 0.2033 - val_binary_accuracy: 0.2859 - val_loss: 0.6183
Epoch 2/2
95/95 ━━━━━━━━━━━━━━━━━━━━ 795s 8s/step - binary_accuracy: 0.5633 - loss: 0.4893 - val_binary_accuracy: 0.6733 - val_loss: 0.4580


Note: I don't think two epochs was enough here. This is something we should go through with CV, plotting accuracy and loss over each iteration

In [16]:
# Fine tune entire model
base_model.trainable = True
model.summary(show_trainable=True)

model.compile(
    optimizer = keras.optimizers.Adam(1e-5), # can also play with learning rate
    loss = keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy()] #check this -- we have a binary model so it should be ok
)

epochs = 1 #??
print("Fitting end-to-end model")
model.fit(training_data, epochs = epochs, validation_data = validation_data)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_1 (InputLayer)  │ (None, 200, 200, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ rescaling (Rescaling)       │ (None, 200, 200, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ xception (Functional)       │ (None, 7, 7, 2048)    │ 20,861,480 │   Y   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ global_average_pooling2d    │ (None, 2048)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout (Dropout)           │ (None, 2048)          │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dense (Dense)               │ (None, 1)             │      2,049 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 20,867,629 (79.60 MB)

 Trainable params: 20,809,001 (79.38 MB)

 Non-trainable params: 54,528 (213.00 KB)

 Optimizer params: 4,100 (16.02 KB)

Fitting end-to-end model
95/95 ━━━━━━━━━━━━━━━━━━━━ 2853s 30s/step - binary_accuracy: 0.9428 - loss: 0.2670 - val_binary_accuracy: 0.8798 - val_loss: 0.3730


In [17]:
# Validate model on test data
print("Test dataset evaluation:")
model.evaluate(testing_data)

Test dataset evaluation:
2/2 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - binary_accuracy: 0.9497 - loss: 0.2773


[0.3067767322063446, 0.9245283007621765]

We now plot loss and accuracy over epochs...